In [1]:
#libraries used
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from sklearn.multiclass import OneVsRestClassifier
import model_helper_functions
from xgboost import XGBClassifier
from catboost import CatBoostClassifier

In [2]:
#load data
data_df = pd.read_csv("data/train.csv")
test_df = pd.read_csv("data/test.csv")
# print(data_df.columns)
# print(test_df.columns)

In [3]:
#pre-processing
data_df = model_helper_functions.feature_engineer(data_df)
    #per eda, drop rows where u < 10
data_df.drop(data_df[data_df['u'] < 10].index, inplace=True)
data_df.drop(columns=['id'], inplace=True)

test_df = model_helper_functions.feature_engineer(test_df)
    #per eda, drop rows where u < 10
test_df.drop(test_df[test_df['u'] < 10].index, inplace=True)
# test_df.drop(columns=['id'], inplace=True)


In [53]:
data_df.columns

Index(['u', 'g', 'r', 'i', 'z', 'redshift', 'spectral_type',
       'galaxy_population', 'class', 'X_loc', 'Y_loc', 'Z_loc', 'u_g_diff',
       'g_r_diff', 'r_i_diff', 'i_z_diff'],
      dtype='object')

In [29]:
#convert categorical variables to numeric dummy variables and do a test-train split
# import importlib
# importlib.reload(model_helper_functions)
if(True):
    cleaned_df = model_helper_functions.cat_to_dummy(data_df,False)
    cleaned_df = cleaned_df.drop(['spectral_type','galaxy_population'],axis=1)
    X_train, X_test, y_train, y_test = train_test_split(cleaned_df.drop(columns=['class']), cleaned_df['class'], test_size=0.25, random_state = 12345)
else: #no need to convert categorical to dummy encoding
    X_train, X_test, y_train, y_test = train_test_split(data_df.drop(columns=['class']), data_df['class'], test_size=0.25, random_state = 12345)

In [15]:
print("Length of training data split: ", len(X_train), "and", len(y_train))
print("Length of testing section: ", len(X_test), "and", len(y_test))

Length of training data split:  433008 and 433008
Length of testing section:  144336 and 144336


In [16]:
cleaned_df.head()

,u,g,r,i,z,redshift,class,X_loc,Y_loc,Z_loc,u_g_diff,g_r_diff,r_i_diff,i_z_diff,A/F,G/K,M,O/B,Blue_Cloud,Red_Sequence
0,25.472123,21.895559,20.357926,19.257113,18.621057,0.408982,0,-0.808809,0.510631,0.291692,3.576564,1.537632,1.100813,0.636056,0,0,1,0,0,1
1,20.778509,19.087062,17.587208,17.226067,16.786433,0.157976,0,-0.519995,0.665835,0.535041,1.691447,1.499854,0.361141,0.439634,0,0,1,0,0,1
2,21.035203,21.079128,21.171840,20.582629,20.557366,2.823770,1,-0.815680,0.002952,0.578496,-0.043925,-0.092712,0.589211,0.025263,0,0,0,1,1,0
3,23.305056,21.050736,19.017754,18.365658,17.914952,0.536099,0,-0.461171,-0.474536,0.749758,2.254320,2.032982,0.652096,0.450706,0,0,1,0,0,1
4,21.703158,19.471680,18.234449,17.899447,17.616185,0.555761,0,-0.741866,0.583034,0.331220,2.231478,1.237231,0.335002,0.283262,0,0,1,0,0,1


In [13]:
###
# XGboost code, copied from https://xgboost.readthedocs.io/en/stable/get_started.html
# Hyperparameter tuning implemented by simple grid search
###
depth_range = range(2, 7, 1)
n_estimators_range = range(1, 10, 1)
learning_rate_range = [0.01, 0.1, 0.5, 1]
for depth in depth_range:
    for n_estimators in n_estimators_range:
        for learning_rate in learning_rate_range:
            bst = XGBClassifier(n_estimators=n_estimators, max_depth=depth, learning_rate=learning_rate,enable_categorical=True)
            # fit model
            bst.fit(X_train, y_train)
            model_helper_functions.save_model_results(bst, X_test, y_test,"xgboost_validation"+str(depth)+"_"+str(n_estimators)+"_"+str(learning_rate),model_description="valid")
            model_helper_functions.save_model_results(bst, X_train, y_train,"xgboost_fit"+str(depth)+"_"+str(n_estimators)+"_"+str(learning_rate),model_description="fit")
            print("Completed depth: ", depth, "n_estimators: ", n_estimators, "learning_rate: ", learning_rate)


Completed depth:  2 n_estimators:  1 learning_rate:  0.01
Completed depth:  2 n_estimators:  1 learning_rate:  0.1
Completed depth:  2 n_estimators:  1 learning_rate:  0.5
Completed depth:  2 n_estimators:  1 learning_rate:  1
Completed depth:  2 n_estimators:  2 learning_rate:  0.01
Completed depth:  2 n_estimators:  2 learning_rate:  0.1
Completed depth:  2 n_estimators:  2 learning_rate:  0.5
Completed depth:  2 n_estimators:  2 learning_rate:  1
Completed depth:  2 n_estimators:  3 learning_rate:  0.01
Completed depth:  2 n_estimators:  3 learning_rate:  0.1
Completed depth:  2 n_estimators:  3 learning_rate:  0.5
Completed depth:  2 n_estimators:  3 learning_rate:  1
Completed depth:  2 n_estimators:  4 learning_rate:  0.01
Completed depth:  2 n_estimators:  4 learning_rate:  0.1
Completed depth:  2 n_estimators:  4 learning_rate:  0.5
Completed depth:  2 n_estimators:  4 learning_rate:  1
Completed depth:  2 n_estimators:  5 learning_rate:  0.01
Completed depth:  2 n_estimators: 

In [14]:
###
# catreg code, grid search hyperparameter tuning
###
for depth in depth_range:
    for learning_rate in learning_rate_range:
        model = CatBoostClassifier(
            iterations=100,  #based on small experiments seems to be sufficient    
            learning_rate=learning_rate,   
            depth=depth,              
            verbose=0,
            cat_features=X_train.select_dtypes(include='category').columns.tolist()             
        )
        model.fit(X_train, y_train)
        model_helper_functions.save_model_results(model, X_test, y_test,"catboost_validation"+str(depth)+"_"+str(learning_rate),model_description="valid")
        model_helper_functions.save_model_results(model, X_train, y_train,"catboost_fit"+str(depth)+"_"+str(learning_rate),model_description="fit")
        print("Completed depth: ", depth, "learning_rate: ", learning_rate)


Completed depth:  2 learning_rate:  0.01
Completed depth:  2 learning_rate:  0.1
Completed depth:  2 learning_rate:  0.5
Completed depth:  2 learning_rate:  1
Completed depth:  3 learning_rate:  0.01
Completed depth:  3 learning_rate:  0.1
Completed depth:  3 learning_rate:  0.5
Completed depth:  3 learning_rate:  1
Completed depth:  4 learning_rate:  0.01
Completed depth:  4 learning_rate:  0.1
Completed depth:  4 learning_rate:  0.5
Completed depth:  4 learning_rate:  1
Completed depth:  5 learning_rate:  0.01
Completed depth:  5 learning_rate:  0.1
Completed depth:  5 learning_rate:  0.5
Completed depth:  5 learning_rate:  1
Completed depth:  6 learning_rate:  0.01
Completed depth:  6 learning_rate:  0.1
Completed depth:  6 learning_rate:  0.5
Completed depth:  6 learning_rate:  1


In [20]:
###
# Logistic regression code
###

first_model = LogisticRegression(solver='lbfgs', max_iter=500)
first_model.fit(X_train, y_train)
# list out the model weights and intercepts for each class category:

# weights = first_model.coef_
# intercepts = first_model.intercept_

# # put all the values into a dataframe with relevant column names
# df_weights = pd.DataFrame(weights, columns=X_train.columns)
# df_weights["intercept"] = intercepts
# df_weights.index.name = "Class"

# print(df_weights)

model_helper_functions.save_model_results(first_model, X_test, y_test,"logreg_validation",model_description="valid")
model_helper_functions.save_model_results(first_model, X_train, y_train,"logreg_fit",model_description="fit")

{'balanced_accuracy': 0.8969663009574392,
 'per_class': {0: {'precision': 0.9498405100869913,
   'recall': 0.9567871559762745,
   'f1': 0.9533011782564059,
   'support': 282902},
  1: {'precision': 0.9330211585154353,
   'recall': 0.9155132510437466,
   'f1': 0.9241842940091849,
   'support': 88144},
  2: {'precision': 0.8241181536061871,
   'recall': 0.8185984958522966,
   'f1': 0.8213490514861265,
   'support': 61962}},
 'macro_precision': 0.9023266074028712,
 'macro_recall': 0.8969663009574392,
 'macro_f1': 0.8996115079172391,
 'weighted_precision': 0.9284262785319466,
 'weighted_recall': 0.9286110187340649,
 'weighted_f1': 0.9284921762887124,
 'matthews_corrcoef': 0.8597937937988799,
 'cohen_kappa': 0.8597531897015012,
 'roc_auc_ovr': 0.9820969305442558,
 'confusion_matrix': array([[270677,   5393,   6832],
        [  3454,  80697,   3993],
        [ 10840,    400,  50722]]),
 'y_pred': array([0, 1, 0, ..., 0, 0, 2], shape=(433008,))}

In [12]:
# Extract weights from all classes in the model
weights = np.array([estimator.coef_[0] for estimator in first_model.estimators_])

# Extract biases (intercepts) for all classes
biases = np.array([estimator.intercept_[0] for estimator in first_model.estimators_])

for target_class, w, b in zip(first_model.classes_, weights, biases):
    print(f"Class {target_class} -> Bias: {b:.4f} | Weights: {w}")

Class 0 -> Bias: 0.0002 | Weights: [ 7.99804608e-07  5.15928829e-03  4.05981495e-03  2.77064879e-03
  2.03760895e-03  1.67533799e-03 -5.52618560e-04 -2.02455326e-04
 -3.37116544e-05  1.82228778e-05  1.09947334e-03  1.28916617e-03
  7.33039840e-04  3.62270954e-04 -4.22256662e-04 -4.69087479e-05
  8.15210116e-04 -1.90909572e-04 -5.81371513e-04  7.36506648e-04]
Class 1 -> Bias: -0.0016 | Weights: [-1.24223596e-06 -5.41395810e-02 -2.39430662e-02  3.39303896e-03
  1.80355142e-02  2.39808444e-02  3.76031137e-02  3.71585198e-03
  5.41419545e-04  1.43866645e-03 -3.01965148e-02 -2.73361052e-02
 -1.46424752e-02 -5.94533015e-03  9.78441252e-03 -9.32909472e-04
 -1.65062522e-02  6.05644611e-03  1.46597686e-02 -1.62580717e-02]
Class 2 -> Bias: -0.0001 | Weights: [ 1.05826121e-07 -2.82475340e-02 -2.66732582e-02 -1.61674882e-02
 -9.65308762e-03 -5.45089791e-03 -1.27299195e-02  2.64239694e-03
  6.32055577e-04 -6.81937489e-04 -1.57427582e-03 -1.05057700e-02
 -6.51440054e-03 -4.20218971e-03  4.56984521e-

In [22]:
###
# Find top three based on validation balanced_accuracy and select best, while looking at in-sample performance and other metrics to ensure reasonable
###
df_results = pd.read_csv("raw_model_results.csv")
# 1) Top 3 rows by balanced_accuracy among rows where model_description == "valid"
valid = df_results[df_results["model_description"] == "valid"]
top_valid = valid.nlargest(3, "balanced_accuracy")
print(top_valid)

# 2) Matching 'fit' rows, in the same order.
# A valid row and its fit row have different model_name values
# (e.g. catboost_validation6_0.5 -> catboost_fit6_0.5), so map the name first.
fit = df_results[df_results["model_description"] == "fit"]
fit_names = top_valid["model_name"].str.replace("validation", "fit", n=1, regex=False)

top_fit = pd.concat([fit[fit["model_name"] == name] for name in fit_names])
print(top_fit)

                   model_name model_description  balanced_accuracy  \
400  catboost_validation6_0.5             valid           0.948270   
402    catboost_validation6_1             valid           0.947203   
392  catboost_validation5_0.5             valid           0.946238   

     macro_precision  macro_recall  macro_f1  weighted_precision  \
400         0.950215      0.948270  0.949236            0.962570   
402         0.948623      0.947203  0.947910            0.961680   
392         0.949369      0.946238  0.947788            0.961608   

     weighted_recall  weighted_f1  matthews_corrcoef  cohen_kappa  \
400         0.962643     0.962603           0.926608     0.926605   
402         0.961735     0.961706           0.924846     0.924844   
392         0.961721     0.961657           0.924747     0.924739   

     roc_auc_ovr                                          per_class  \
400     0.995391  {"0": {"precision": 0.9731375736209917, "recal...   
402     0.995022  {"0": {"p

In [28]:
###
# Isolate fit best model, catboost_fit6_0.5
###

model = CatBoostClassifier(
        iterations=100,  #based on small experiments seems to be sufficient    
        learning_rate=0.5,   
        depth=6,              
        verbose=0,
        cat_features=X_train.select_dtypes(include='category').columns.tolist()             
    )
model.fit(X_train, y_train)

CatBoostClassifier(cat_features=['A/F', 'G/K', 'M', 'O/B', 'Blue_Cloud', 'Red_Sequence'], depth=6, iterations=100, learning_rate=0.5, verbose=0)

In [25]:
#clean test data a lil
test_df = model_helper_functions.cat_to_dummy(test_df,False)
test_df.drop(['spectral_type','galaxy_population'],axis=1)

In [16]:
print(test_df.columns)
print(X_train.columns)

Index(['id', 'u', 'g', 'r', 'i', 'z', 'redshift', 'X_loc', 'Y_loc', 'Z_loc',
       'u_g_diff', 'g_r_diff', 'r_i_diff', 'i_z_diff', 'A/F', 'G/K', 'M',
       'O/B', 'Blue_Cloud', 'Red_Sequence', 'spectral_type',
       'galaxy_population'],
      dtype='object')
Index(['u', 'g', 'r', 'i', 'z', 'redshift', 'X_loc', 'Y_loc', 'Z_loc',
       'u_g_diff', 'g_r_diff', 'r_i_diff', 'i_z_diff', 'A/F', 'G/K', 'M',
       'O/B', 'Blue_Cloud', 'Red_Sequence', 'spectral_type',
       'galaxy_population'],
      dtype='object')


In [18]:
test_df_without_id = test_df.drop(columns=['id'])
final_predicts = model.predict(test_df_without_id)


In [47]:
# print(list(final_predicts)))
# print(final_predicts.shape)
# print(pd.Series(final_predicts[:,0]).head())
# print(len(test_df))
model_df = pd.DataFrame({'id': test_df['id'], 'class': pd.Series(final_predicts[:,0])})
model_df = model_helper_functions.dummy_to_cat(model_df)

model_df.to_csv("submissions/final_predictions.csv", index=False)

In [46]:
model_df.head()

,id,class
0,577347,GALAXY
1,577348,GALAXY
2,577349,GALAXY
3,577350,STAR
4,577351,GALAXY


## Feature importance for best model

In [ ]:
model

In [ ]:
###
# later wrap into a function, getting final result output for the test data set###


# set up dummy variables to ensure everything is numerical for the model
spectral = pd.get_dummies(test_df['spectral_type'], dtype=int)
galaxy_pop = pd.get_dummies(test_df['galaxy_population'], dtype=int)
    
# drop original categorical columns and concatenate the dummy variables to feed into logistic regression
temp_df = test_df.drop(columns=['spectral_type', 'galaxy_population'])
fin_test_df = pd.concat([temp_df, spectral, galaxy_pop], axis=1)

In [24]:
print(fin_test_df.head())

       id          u          g          r          i          z  redshift  \
0  577347  23.668066  21.951680  21.086183  20.180032  19.202124  0.429042   
1  577348  24.902933  22.338822  20.732163  19.860330  19.687691  0.867305   
2  577349  19.427591  18.474633  17.551314  16.570674  16.176765  0.224234   
3  577350  23.121029  21.526855  20.670159  20.417633  20.699095  0.066507   
4  577351  25.094282  22.643981  21.123173  19.439500  19.094158  0.977218   

      X_loc     Y_loc     Z_loc  u_g_diff  g_r_diff  r_i_diff  i_z_diff  A/F  \
0 -0.466950  0.785815  0.405528  1.716387  0.865497  0.906151  0.977908    0   
1 -0.572583 -0.470566  0.671354  2.564112  1.606658  0.871833  0.172640    0   
2 -0.993240  0.111959 -0.030650  0.952958  0.923319  0.980640  0.393909    0   
3 -0.996037 -0.085460 -0.024625  1.594174  0.856696  0.252526 -0.281462    0   
4 -0.711008 -0.651240  0.265243  2.450301  1.520808  1.683673  0.345342    0   

   G/K  M  O/B  Blue_Cloud  Red_Sequence  
0    1 

In [ ]:
# predict the classes of the testing data
final_predicts = first_model.predict(fin_test_df)


In [ ]:
# put the results into a DataFrame with 'id', can save to a csv and submit if needed

logis_df = pd.DataFrame({'id': fin_test_df['id'], 'class': final_predicts})
logis_df = model_helper_functions.dummy_to_cat(logis_df)

In [ ]:
# verify the resulting df looks correct
print(logis_df.head(20))

        id   class
0   577347  GALAXY
1   577348  GALAXY
2   577349  GALAXY
3   577350    STAR
4   577351  GALAXY
5   577352  GALAXY
6   577353  GALAXY
7   577354    STAR
8   577355  GALAXY
9   577356  GALAXY
10  577357    STAR
11  577358  GALAXY
12  577359     QSO
13  577360  GALAXY
14  577361     QSO
15  577362     QSO
16  577363    STAR
17  577364    STAR
18  577365     QSO
19  577366  GALAXY


## For our best model, let us look into feature selection

In [30]:
#rerun test-train split, as there was some earlier overloading of _test
if(True):
    cleaned_df = model_helper_functions.cat_to_dummy(data_df,False)
    cleaned_df = cleaned_df.drop(['spectral_type','galaxy_population'],axis=1)
    X_train, X_test, y_train, y_test = train_test_split(cleaned_df.drop(columns=['class']), cleaned_df['class'], test_size=0.25, random_state = 12345)
else: #no need to convert categorical to dummy encoding
    X_train, X_test, y_train, y_test = train_test_split(data_df.drop(columns=['class']), data_df['class'], test_size=0.25, random_state = 12345)

In [37]:
from catboost import Pool

cat_cols = X_test.select_dtypes(include='category').columns.tolist()
pool = Pool(X_test, label=y_test, cat_features=cat_cols)
sv = model.get_feature_importance(data=pool, type='ShapValues')   # shape (n, p+1)

shap_by_class = {
    cls: pd.DataFrame(sv[:, k, :-1], columns=X_test.columns, index=X_test.index)
    for k, cls in enumerate(model.classes_)
}
expected_values = dict(zip(model.classes_, sv[0, :, -1]))

importance = pd.Series(np.abs(sv[:, :, :-1]).mean(axis=(0, 1)),
                       index=X_test.columns).sort_values(ascending=False)

In [40]:
long = pd.DataFrame(sv[:, :, :-1].reshape(-1, X_test.shape[1]), columns=X_test.columns,
                    index=pd.MultiIndex.from_product([X_test.index, model.classes_],
                                                     names=['row', 'class']))

In [42]:
importance

redshift        1.570082
g_r_diff        0.385825
r_i_diff        0.372348
u_g_diff        0.319681
z               0.311284
u               0.220751
g               0.184295
X_loc           0.168789
Z_loc           0.165270
Y_loc           0.155722
i_z_diff        0.147839
r               0.142643
i               0.118952
Blue_Cloud      0.068246
M               0.056762
A/F             0.018516
O/B             0.015651
G/K             0.014948
Red_Sequence    0.014462
dtype: float64